# M3: LSTM with Entity Embeddings

This notebook implements an LSTM-based model for next-day retail sales
forecasting using 30-day historical sequences and learned entity embeddings
for Store, StoreType, Assortment, and StateHoliday.

The model is developed using the common Rossmann preprocessing pipeline
prepared for the project.

## 1. Data Loading and Preparation

The common preprocessed Rossmann training and validation datasets are loaded
and combined with store-level metadata from `store.csv`.

The store metadata provides the categorical variables required for the
entity embeddings.

In [49]:
import pandas as pd

store = pd.read_csv("data/store.csv")

print("Loaded successfully!")
print("Shape:", store.shape)
print("Columns:", store.columns.tolist())

display(store.head())

Loaded successfully!
Shape: (1115, 10)
Columns: ['Store', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']


,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [50]:
train = pd.read_csv("data/processed/rossmann_train.csv")
validation = pd.read_csv("data/processed/rossmann_validation.csv")

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)

Train shape: (949194, 10)
Validation shape: (68015, 10)


In [51]:
train_lstm = train.merge(store, on="Store", how="left")
validation_lstm = validation.merge(store, on="Store", how="left")

print("LSTM Train shape:", train_lstm.shape)
print("LSTM Validation shape:", validation_lstm.shape)

LSTM Train shape: (949194, 19)
LSTM Validation shape: (68015, 19)


In [52]:
print("Missing StoreType:", train_lstm["StoreType"].isna().sum())
print("Missing Assortment:", train_lstm["Assortment"].isna().sum())
print("Missing StateHoliday:", train_lstm["StateHoliday"].isna().sum())

Missing StoreType: 0
Missing Assortment: 0
Missing StateHoliday: 0


## 2. Categorical Entity Encoding

The categorical variables `Store`, `StoreType`, `Assortment`, and
`StateHoliday` are converted into integer IDs.

These IDs are passed to learned embedding layers rather than being
one-hot encoded.

In [53]:
categorical_cols = [
    "Store",
    "StoreType",
    "Assortment",
    "StateHoliday"
]

for col in categorical_cols:
    categories = train_lstm[col].unique()
    
    mapping = {value: idx for idx, value in enumerate(categories)}
    
    train_lstm[col + "_idx"] = train_lstm[col].map(mapping)
    validation_lstm[col + "_idx"] = validation_lstm[col].map(mapping)

print("Encoding complete.")

Encoding complete.


In [54]:
train_lstm["Date"] = pd.to_datetime(train_lstm["Date"])
validation_lstm["Date"] = pd.to_datetime(validation_lstm["Date"])

print(train_lstm["Date"].dtype)
print(validation_lstm["Date"].dtype)

datetime64[ns]
datetime64[ns]


In [55]:
numeric_cols = ["DayOfWeek", "Promo", "Open", "SchoolHoliday"]

for col in numeric_cols:
    train_lstm[col] = pd.to_numeric(train_lstm[col], errors="coerce")
    validation_lstm[col] = pd.to_numeric(validation_lstm[col], errors="coerce")

print(train_lstm[numeric_cols].dtypes)
print("\nMissing values:")
print(train_lstm[numeric_cols].isna().sum())

for col in numeric_cols:
    print(col, ":", sorted(train_lstm[col].unique()))

DayOfWeek        int64
Promo            int64
Open             int64
SchoolHoliday    int64
dtype: object

Missing values:
DayOfWeek        0
Promo            0
Open             0
SchoolHoliday    0
dtype: int64
DayOfWeek : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Promo : [np.int64(0), np.int64(1)]
Open : [np.int64(0), np.int64(1)]
SchoolHoliday : [np.int64(0), np.int64(1)]


## 3. Temporal Sequence Construction

A 30-day lookback window is used for next-day sales forecasting.

Sequences are created in chronological order for each store. Time-series
gaps are handled using segments so that a sequence does not cross a gap
in the historical data.

The target is the sales value of the day immediately following the
30-day input sequence.

In [56]:
# Mark where each row comes from
train_lstm["_split"] = "train"
validation_lstm["_split"] = "validation"

# Combine train + validation so validation can use previous training history
combined = pd.concat(
    [train_lstm, validation_lstm],
    ignore_index=True
)

# Make sure chronological order is correct for every store
combined = combined.sort_values(
    ["Store", "Date"]
).reset_index(drop=True)

# Detect gaps again on the combined data
combined["date_diff"] = (
    combined.groupby("Store")["Date"]
    .diff()
    .dt.days
)

# Start a new segment whenever there is a date gap
combined["segment_id"] = (
    (combined["date_diff"] != 1)
    .groupby(combined["Store"])
    .cumsum()
)

combined.drop(columns=["date_diff"], inplace=True)

print("Combined shape:", combined.shape)
print("Train rows:", (combined["_split"] == "train").sum())
print("Validation rows:", (combined["_split"] == "validation").sum())
print("Segments:", combined["segment_id"].nunique())

Combined shape: (1017209, 25)
Train rows: 949194
Validation rows: 68015
Segments: 2


In [57]:
import numpy as np

LOOKBACK = 30

def create_sequence_indices(df, target_split, lookback=30):
    starts = []
    targets = []

    for (_, segment_id), group in df.groupby(
        ["Store", "segment_id"],
        sort=False
    ):
        indices = group.index.to_numpy()

        # Target positions belonging to the requested split
        target_positions = np.where(
            group["_split"].to_numpy() == target_split
        )[0]

        for pos in target_positions:
            # Need 30 previous consecutive days
            if pos >= lookback:
                starts.append(indices[pos - lookback])
                targets.append(indices[pos])

    return (
        np.asarray(starts, dtype=np.int64),
        np.asarray(targets, dtype=np.int64)
    )

In [58]:
train_starts, train_targets = create_sequence_indices(
    combined,
    target_split="train",
    lookback=LOOKBACK
)

val_starts, val_targets = create_sequence_indices(
    combined,
    target_split="validation",
    lookback=LOOKBACK
)

print("Training sequences:", len(train_targets))
print("Validation sequences:", len(val_targets))

Training sequences: 910344
Validation sequences: 68015


In [59]:
import numpy as np

# Features used at each timestep
sequence_features = [
    "Sales",
    "DayOfWeek",
    "Open",
    "Promo",
    "SchoolHoliday"
]

categorical_features = [
    "Store_idx",
    "StoreType_idx",
    "Assortment_idx",
    "StateHoliday_idx"
]

# Copy only the arrays needed by the model
numeric_array = combined[sequence_features].to_numpy(dtype=np.float32)
categorical_array = combined[categorical_features].to_numpy(dtype=np.int64)
sales_array = combined["Sales"].to_numpy(dtype=np.float32)

# Use log1p for the forecasting target and historical sales input
numeric_array[:, 0] = np.log1p(numeric_array[:, 0])
sales_array_log = np.log1p(sales_array)

print("Numeric array:", numeric_array.shape)
print("Categorical array:", categorical_array.shape)
print("Target array:", sales_array_log.shape)

Numeric array: (1017209, 5)
Categorical array: (1017209, 4)
Target array: (1017209,)


## 4. PyTorch Dataset and DataLoader

The prepared numerical and categorical features are converted into a
PyTorch Dataset and DataLoader for mini-batch training.

In [60]:
import torch
from torch.utils.data import Dataset, DataLoader


class RossmannLSTMDataset(Dataset):
    def __init__(
        self,
        starts,
        targets,
        numeric_array,
        categorical_array,
        sales_array_log,
        lookback=30
    ):
        self.starts = starts
        self.targets = targets
        self.numeric_array = numeric_array
        self.categorical_array = categorical_array
        self.sales_array_log = sales_array_log
        self.lookback = lookback

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        start = self.starts[idx]
        target_idx = self.targets[idx]

        end = start + self.lookback

        # Numerical sequence
        x_numeric = self.numeric_array[start:end].copy()

        # Replace raw Sales with log1p(Sales)
        # numeric_array[:, 0] is already log-transformed above

        # Categorical sequence
        x_categorical = self.categorical_array[start:end]

        # Next-day target
        y = self.sales_array_log[target_idx]

        return (
            torch.tensor(x_numeric, dtype=torch.float32),
            torch.tensor(x_categorical, dtype=torch.long),
            torch.tensor(y, dtype=torch.float32)
        )

In [61]:
train_dataset = RossmannLSTMDataset(
    train_starts,
    train_targets,
    numeric_array,
    categorical_array,
    sales_array_log,
    lookback=LOOKBACK
)

val_dataset = RossmannLSTMDataset(
    val_starts,
    val_targets,
    numeric_array,
    categorical_array,
    sales_array_log,
    lookback=LOOKBACK
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

Train dataset: 910344
Validation dataset: 68015


## 5. LSTM with Entity Embeddings

The model uses learned embeddings for:

- Store
- StoreType
- Assortment
- StateHoliday

The embedding representations are concatenated with the numerical
time-series features and passed through an LSTM.

The final LSTM representation is used to predict next-day sales.

In [62]:
import torch
import torch.nn as nn


class LSTMEntityModel(nn.Module):
    def __init__(
        self,
        store_vocab=1115,
        store_type_vocab=4,
        assortment_vocab=3,
        holiday_vocab=4,
        num_numeric_features=5,
        store_emb_dim=16,
        store_type_emb_dim=2,
        assortment_emb_dim=2,
        holiday_emb_dim=2,
        hidden_size=64,
        num_layers=1,
        dropout=0.2
    ):
        super().__init__()

        # Entity embeddings
        self.store_embedding = nn.Embedding(
            store_vocab, store_emb_dim
        )

        self.store_type_embedding = nn.Embedding(
            store_type_vocab, store_type_emb_dim
        )

        self.assortment_embedding = nn.Embedding(
            assortment_vocab, assortment_emb_dim
        )

        self.holiday_embedding = nn.Embedding(
            holiday_vocab, holiday_emb_dim
        )

        # Total input size after concatenation
        input_size = (
            num_numeric_features
            + store_emb_dim
            + store_type_emb_dim
            + assortment_emb_dim
            + holiday_emb_dim
        )

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0 if num_layers == 1 else dropout
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x_numeric, x_categorical):

        store_id = x_categorical[:, :, 0]
        store_type_id = x_categorical[:, :, 1]
        assortment_id = x_categorical[:, :, 2]
        holiday_id = x_categorical[:, :, 3]

        # Create embeddings
        store_emb = self.store_embedding(store_id)
        store_type_emb = self.store_type_embedding(store_type_id)
        assortment_emb = self.assortment_embedding(assortment_id)
        holiday_emb = self.holiday_embedding(holiday_id)

        # Concatenate numerical + categorical embeddings
        x = torch.cat(
            [
                x_numeric,
                store_emb,
                store_type_emb,
                assortment_emb,
                holiday_emb
            ],
            dim=-1
        )

        # LSTM
        output, _ = self.lstm(x)

        # Last timestep
        last_output = output[:, -1, :]

        last_output = self.dropout(last_output)

        # Prediction
        prediction = self.fc(last_output)

        return prediction.squeeze(-1)

In [63]:
from torch.utils.data import DataLoader

BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train batches: 7113
Validation batches: 532


## 6. Preliminary Training

The model is trained using the Adam optimizer and mean squared error
loss on the log-transformed sales target.

This is an initial 3-epoch baseline run and has not yet undergone
hyperparameter optimization.

In [64]:
import time
import torch
import torch.nn as nn
import torch.optim as optim

# Fresh baseline model
model = LSTMEntityModel()

criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

NUM_EPOCHS = 1

best_val_loss = float("inf")

print("Starting baseline training...")
start_time = time.time()

for epoch in range(NUM_EPOCHS):

    # -------------------
    # Training
    # -------------------
    model.train()

    train_loss = 0.0

    for batch_idx, (x_num, x_cat, y) in enumerate(train_loader):

        optimizer.zero_grad()

        predictions = model(x_num, x_cat)

        loss = criterion(predictions, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        if (batch_idx + 1) % 1000 == 0:
            print(
                f"  Batch {batch_idx + 1}/{len(train_loader)}"
            )

    train_loss /= len(train_loader)

    # -------------------
    # Validation
    # -------------------
    model.eval()

    val_loss = 0.0

    with torch.no_grad():
        for x_num, x_cat, y in val_loader:

            predictions = model(x_num, x_cat)

            loss = criterion(predictions, y)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    elapsed = time.time() - start_time

    print(
        f"\nEpoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Time: {elapsed / 60:.2f} min"
    )

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "lstm_baseline_best.pt"
        )

        print("Best model saved.")

Starting baseline training...
  Batch 1000/7113
  Batch 2000/7113
  Batch 3000/7113
  Batch 4000/7113
  Batch 5000/7113
  Batch 6000/7113
  Batch 7000/7113

Epoch 1/1 | Train Loss: 3.4251 | Val Loss: 0.6237 | Time: 7.24 min
Best model saved.


In [65]:
import time

additional_epochs = 2
best_val_loss = float("inf")

epoch_results = []

start_time = time.time()

for epoch in range(additional_epochs):

    model.train()
    train_loss = 0.0

    for x_num, x_cat, y in train_loader:
        optimizer.zero_grad()

        predictions = model(x_num, x_cat)
        loss = criterion(predictions, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for x_num, x_cat, y in val_loader:
            predictions = model(x_num, x_cat)
            loss = criterion(predictions, y)
            val_loss += loss.item()

    val_loss /= len(val_loader)

    elapsed = time.time() - start_time

    epoch_number = epoch + 2

    epoch_results.append({
        "Epoch": epoch_number,
        "Train Loss": train_loss,
        "Val Loss": val_loss
    })

    print(
        f"Epoch {epoch_number}/3 | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Time: {elapsed/60:.2f} min"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "lstm_baseline_best.pt"
        )

        print("Best model saved.")

Epoch 2/3 | Train Loss: 1.4817 | Val Loss: 0.6312 | Time: 8.97 min
Best model saved.
Epoch 3/3 | Train Loss: 1.1609 | Val Loss: 0.6665 | Time: 18.05 min


In [66]:
model.load_state_dict(
    torch.load("lstm_baseline_best.pt", map_location="cpu")
)

model.eval()

print("Best checkpoint loaded.")

Best checkpoint loaded.


## 7. Preliminary Validation Results

The model is evaluated on the validation period using RMSE, MAE, and
RMSPE after converting predictions back to the original sales scale.

These results are preliminary and will be further refined during the
final optimization phase.

In [67]:
model.load_state_dict(
    torch.load("lstm_baseline_best.pt", map_location="cpu")
)

model.eval()

all_predictions = []
all_targets = []

with torch.no_grad():
    for x_num, x_cat, y in val_loader:
        predictions = model(x_num, x_cat)

        all_predictions.append(predictions.numpy())
        all_targets.append(y.numpy())

pred_log = np.concatenate(all_predictions)
true_log = np.concatenate(all_targets)

pred_sales = np.maximum(np.expm1(pred_log), 0)
true_sales = np.expm1(true_log)

rmse = np.sqrt(np.mean((true_sales - pred_sales) ** 2))
mae = np.mean(np.abs(true_sales - pred_sales))

nonzero = true_sales > 0

rmspe = np.sqrt(
    np.mean(
        ((true_sales[nonzero] - pred_sales[nonzero])
         / true_sales[nonzero]) ** 2
    )
) * 100

print("M3 LSTM + Entity Embeddings — 3 Epoch Preliminary Result")
print("-" * 55)
print(f"RMSE : {rmse:,.2f}")
print(f"MAE  : {mae:,.2f}")
print(f"RMSPE: {rmspe:.2f}%")
print(f"Validation samples: {len(true_sales):,}")

M3 LSTM + Entity Embeddings — 3 Epoch Preliminary Result
-------------------------------------------------------
RMSE : 2,618.77
MAE  : 1,692.07
RMSPE: 37.31%
Validation samples: 68,015


## 8. Future Work

Future work will include hyperparameter tuning, feature refinement, and
an ablation study to quantify the contribution of entity embeddings.